# MAE 6246 - Week 3 Python Example

## Solutions of Continuous-Time Linear Systems

**Lecture:** Friday, September 11, 2026

This notebook compares several representations of the same linear-system solution. It also constructs an LTV state-transition matrix numerically and demonstrates transient amplification in a stable non-normal system.

## Learning goals

After working through the notebook, you should be able to:

1. compute a homogeneous LTI response with a matrix exponential;
2. verify that matrix-exponential and numerical-integration solutions agree;
3. separate zero-input and zero-state responses;
4. evaluate the convolution form of a forced response;
5. check identity, inverse, and composition properties of a state-transition matrix;
6. construct an LTV state-transition matrix by integrating a matrix differential equation; and
7. distinguish asymptotic decay from finite-time transient amplification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import quad_vec, solve_ivp
from scipy.linalg import expm, norm, svdvals
from scipy.signal import StateSpace, lsim

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "lines.linewidth": 2.0,
})

## 1. Running example: a damped oscillator

Consider a unit-mass oscillator

$$
\ddot q+0.6\dot q+2q=u.
$$

With $x=[q,\dot q]^T$ and output $y=q$,

$$
\dot x=Ax+Bu,\qquad y=Cx+Du,
$$

where

$$
A=\begin{bmatrix}0&1\\-2&-0.6\end{bmatrix},
\quad
B=\begin{bmatrix}0\\1\end{bmatrix},
\quad
C=\begin{bmatrix}1&0\end{bmatrix},
\quad D=0.
$$

This stable, underdamped mechanical system will be used to compare solution methods.

In [ ]:
A = np.array([[0.0, 1.0],
              [-2.0, -0.6]])
B = np.array([[0.0],
              [1.0]])
C = np.array([[1.0, 0.0]])
D = np.array([[0.0]])

eigenvalues = np.linalg.eigvals(A)
print("Eigenvalues of A:", eigenvalues)

## 2. Homogeneous response: matrix exponential versus ODE integration

For $u=0$ and $x(0)=x_0$,

$$
x(t)=e^{At}x_0.
$$

We compute this expression directly with **expm**, then solve the same differential equation with **solve_ivp**.

In [ ]:
t = np.linspace(0.0, 12.0, 601)
x0 = np.array([1.0, -0.25])

x_expm = np.array([expm(A * ti) @ x0 for ti in t])

homogeneous_solution = solve_ivp(
    lambda time, state: A @ state,
    (t[0], t[-1]),
    x0,
    t_eval=t,
    rtol=1e-10,
    atol=1e-12,
)
x_ivp = homogeneous_solution.y.T

print("Maximum state error:", np.max(np.abs(x_expm - x_ivp)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
labels = [r"$q(t)$", r"$\dot q(t)$"]

for index, ax in enumerate(axes):
    ax.plot(t, x_expm[:, index], label="matrix exponential")
    ax.plot(t, x_ivp[:, index], "--", label="solve_ivp")
    ax.set_ylabel(labels[index])
    ax.legend()

axes[-1].set_xlabel("time")
fig.suptitle("Homogeneous response computed two ways")
fig.tight_layout()
plt.show()

The two curves should overlap to plotting accuracy. The matrix exponential is an analytical representation of the solution; **solve_ivp** is a numerical approximation obtained by stepping through the differential equation. Agreement is a useful implementation check, but it does not make the two methods conceptually identical.

## 3. Zero-input and zero-state decomposition

Use the time-varying input

$$
u(t)=1+0.4\sin(1.3t).
$$

Linearity implies

$$
x(t)=x_{\mathrm{zi}}(t)+x_{\mathrm{zs}}(t),
$$

where

$$
x_{\mathrm{zi}}(t)=e^{At}x_0
$$

and

$$
x_{\mathrm{zs}}(t)=\int_0^t e^{A(t-\tau)}Bu(\tau)\,d\tau.
$$

We integrate the complete response and the zero-state response separately, then verify the decomposition.

In [ ]:
def input_signal(time):
    return 1.0 + 0.4 * np.sin(1.3 * time)

def forced_rhs(time, state):
    return A @ state + B[:, 0] * input_signal(time)

def integrate_response(initial_state):
    return solve_ivp(
        forced_rhs,
        (t[0], t[-1]),
        initial_state,
        t_eval=t,
        dense_output=True,
        rtol=1e-10,
        atol=1e-12,
    )

total_solution = integrate_response(x0)
zero_state_solution = integrate_response(np.zeros(2))

x_total = total_solution.y.T
x_zero_state = zero_state_solution.y.T
x_zero_input = np.array([expm(A * ti) @ x0 for ti in t])

decomposition_error = x_total - (x_zero_input + x_zero_state)
print("Maximum decomposition error:", np.max(np.abs(decomposition_error)))

u_values = np.array([input_signal(ti) for ti in t])

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axes[0].plot(t, u_values, color="black")
axes[0].set_ylabel(r"$u(t)$")
axes[0].set_title("Forced response and its decomposition")

state_labels = [r"$q(t)$", r"$\dot q(t)$"]
for index, ax in enumerate(axes[1:]):
    ax.plot(t, x_total[:, index], label="total")
    ax.plot(t, x_zero_input[:, index], "--", label="zero-input")
    ax.plot(t, x_zero_state[:, index], ":", label="zero-state")
    ax.set_ylabel(state_labels[index])
    ax.legend(ncol=3)

axes[-1].set_xlabel("time")
fig.tight_layout()
plt.show()

The zero-input response decays because it contains only the stable homogeneous dynamics. The zero-state response persists because the input continues to act. Their sum reproduces the total response.

## 4. Input-response simulation with scipy.signal.lsim

The same system can be simulated using the standard state-space input-response routine **lsim**. The input must be sampled on a uniform time grid.

In [ ]:
system = StateSpace(A, B, C, D)
t_lsim, y_lsim, x_lsim = lsim(system, U=u_values, T=t, X0=x0)

print("Maximum difference from solve_ivp:",
      np.max(np.abs(x_lsim - x_total)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for index, ax in enumerate(axes):
    ax.plot(t, x_total[:, index], label="solve_ivp")
    ax.plot(t_lsim, x_lsim[:, index], "--", label="lsim")
    ax.set_ylabel(state_labels[index])
    ax.legend()

axes[-1].set_xlabel("time")
fig.suptitle("Forced response computed by two simulation routines")
fig.tight_layout()
plt.show()

## 5. Direct evaluation of the convolution integral

To connect the code to the variation-of-constants formula, evaluate

$$
\int_0^t e^{A(t-\tau)}Bu(\tau)\,d\tau
$$

with numerical quadrature. This is much less efficient than a dedicated simulation routine when it is repeated for many times, but it makes the mathematical structure explicit.

In [ ]:
sample_times = np.linspace(0.0, 12.0, 121)
x_convolution = []

for current_time in sample_times:
    if current_time == 0.0:
        forced_part = np.zeros(2)
    else:
        forced_part, _ = quad_vec(
            lambda tau: expm(A * (current_time - tau))
                        @ B[:, 0] * input_signal(tau),
            0.0,
            current_time,
        )

    homogeneous_part = expm(A * current_time) @ x0
    x_convolution.append(homogeneous_part + forced_part)

x_convolution = np.array(x_convolution)
x_reference = total_solution.sol(sample_times).T

print("Maximum convolution error:",
      np.max(np.abs(x_convolution - x_reference)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for index, ax in enumerate(axes):
    ax.plot(sample_times, x_reference[:, index], label="solve_ivp")
    ax.plot(sample_times, x_convolution[:, index], "o",
            ms=3, label="convolution quadrature")
    ax.set_ylabel(state_labels[index])
    ax.legend()

axes[-1].set_xlabel("time")
fig.suptitle("Variation-of-constants formula evaluated directly")
fig.tight_layout()
plt.show()

## 6. LTI state-transition properties

For an LTI system,

$$
\Phi(t,t_0)=e^{A(t-t_0)}.
$$

We check three defining properties numerically:

$$
\Phi(t_0,t_0)=I,
$$

$$
\Phi(t_2,t_0)=\Phi(t_2,t_1)\Phi(t_1,t_0),
$$

and

$$
\Phi(t,t_0)^{-1}=\Phi(t_0,t).
$$

In [ ]:
def phi_lti(final_time, initial_time):
    return expm(A * (final_time - initial_time))

t0, t1, t2 = 0.4, 1.7, 4.1

identity_error = norm(phi_lti(t0, t0) - np.eye(2))
composition_error = norm(
    phi_lti(t2, t0) - phi_lti(t2, t1) @ phi_lti(t1, t0)
)
inverse_error = norm(
    phi_lti(t0, t2) @ phi_lti(t2, t0) - np.eye(2)
)

print("Identity error:   ", identity_error)
print("Composition error:", composition_error)
print("Inverse error:    ", inverse_error)

## 7. A defective matrix and its exponential

For

$$
A_J=\begin{bmatrix}-1&1\\0&-1\end{bmatrix},
$$

the repeated eigenvalue is $-1$ and the matrix is defective. Its exponential is

$$
e^{A_Jt}=e^{-t}\begin{bmatrix}1&t\\0&1\end{bmatrix}.
$$

The polynomial factor is a signature of the Jordan block.

In [ ]:
A_jordan = np.array([[-1.0, 1.0],
                     [ 0.0, -1.0]])

jordan_error = []
for current_time in t:
    exact = np.exp(-current_time) * np.array(
        [[1.0, current_time],
         [0.0, 1.0]]
    )
    jordan_error.append(norm(expm(A_jordan * current_time) - exact))

print("Maximum Jordan-form exponential error:", max(jordan_error))

x0_jordan = np.array([0.0, 1.0])
x_jordan = np.array([expm(A_jordan * ti) @ x0_jordan for ti in t])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, x_jordan[:, 0], label=r"$x_1(t)=t e^{-t}$")
ax.plot(t, x_jordan[:, 1], label=r"$x_2(t)=e^{-t}$")
ax.set_xlabel("time")
ax.set_ylabel("state")
ax.set_title("Response associated with a Jordan block")
ax.legend()
plt.show()

## 8. Numerically constructing an LTV state-transition matrix

For a time-varying matrix, the shortcut

$$
\Phi(t,t_0)=e^{A(t-t_0)}
$$

does not apply. Instead, integrate the matrix differential equation

$$
\frac{\partial}{\partial t}\Phi(t,t_0)
=A(t)\Phi(t,t_0),
\qquad
\Phi(t_0,t_0)=I.
$$

Use the example

$$
A(t)=
\begin{bmatrix}
-0.3 & 1+0.4\sin t\\
0 & -1
\end{bmatrix}.
$$

In [ ]:
def A_ltv(time):
    return np.array([
        [-0.3, 1.0 + 0.4 * np.sin(time)],
        [ 0.0, -1.0],
    ])

def state_transition_ltv(final_time, initial_time):
    def matrix_rhs(time, flattened_phi):
        phi = flattened_phi.reshape(2, 2)
        return (A_ltv(time) @ phi).reshape(-1)

    solution = solve_ivp(
        matrix_rhs,
        (initial_time, final_time),
        np.eye(2).reshape(-1),
        rtol=1e-10,
        atol=1e-12,
    )
    return solution.y[:, -1].reshape(2, 2)

t0, t1, t2 = 0.0, 1.5, 3.5
phi_20 = state_transition_ltv(t2, t0)
phi_21 = state_transition_ltv(t2, t1)
phi_10 = state_transition_ltv(t1, t0)
phi_02 = state_transition_ltv(t0, t2)

print("LTV composition error:",
      norm(phi_20 - phi_21 @ phi_10))
print("LTV inverse error:",
      norm(phi_02 @ phi_20 - np.eye(2)))

In [ ]:
# Plot every entry of Phi(t, 0).
t_phi = np.linspace(0.0, 6.0, 301)

def matrix_rhs(time, flattened_phi):
    phi = flattened_phi.reshape(2, 2)
    return (A_ltv(time) @ phi).reshape(-1)

phi_solution = solve_ivp(
    matrix_rhs,
    (t_phi[0], t_phi[-1]),
    np.eye(2).reshape(-1),
    t_eval=t_phi,
    rtol=1e-10,
    atol=1e-12,
)
phi_history = phi_solution.y.T.reshape(-1, 2, 2)

fig, ax = plt.subplots(figsize=(8, 4.5))
for row in range(2):
    for column in range(2):
        ax.plot(
            t_phi,
            phi_history[:, row, column],
            label=rf"$\Phi_{{{row+1}{column+1}}}(t,0)$",
        )

ax.set_xlabel("time")
ax.set_ylabel("matrix entry")
ax.set_title("Numerically integrated LTV state-transition matrix")
ax.legend(ncol=2)
plt.show()

## 9. Stable eigenvalues do not guarantee monotone decay

Compare two matrices with the same eigenvalues:

$$
A_{\mathrm{normal}}=
\begin{bmatrix}-1&0\\0&-2\end{bmatrix},
\qquad
A_{\mathrm{non}}=
\begin{bmatrix}-1&8\\0&-2\end{bmatrix}.
$$

Both systems are asymptotically stable. The second matrix is non-normal, however, and its non-orthogonal directions can combine to amplify the state before it decays.

The largest possible gain from a unit initial state at time $t$ is

$$
\|e^{At}\|_2=\sigma_{\max}(e^{At}).
$$

In [ ]:
A_normal = np.diag([-1.0, -2.0])
A_non_normal = np.array([[-1.0, 8.0],
                         [ 0.0, -2.0]])

t_gain = np.linspace(0.0, 6.0, 601)
gain_normal = np.array([
    svdvals(expm(A_normal * ti))[0] for ti in t_gain
])
gain_non_normal = np.array([
    svdvals(expm(A_non_normal * ti))[0] for ti in t_gain
])

peak_index = np.argmax(gain_non_normal)
peak_time = t_gain[peak_index]
peak_gain = gain_non_normal[peak_index]

_, _, right_vectors_transpose = np.linalg.svd(
    expm(A_non_normal * peak_time)
)
optimal_initial_state = right_vectors_transpose[0, :]

x_normal = np.array([
    expm(A_normal * ti) @ optimal_initial_state for ti in t_gain
])
x_non_normal = np.array([
    expm(A_non_normal * ti) @ optimal_initial_state for ti in t_gain
])

print("Eigenvalues:", np.linalg.eigvals(A_non_normal))
print("Peak non-normal gain:", peak_gain)
print("Peak time:", peak_time)
print("Unit initial state producing the peak:", optimal_initial_state)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(t_gain, gain_normal, label="normal")
axes[0].plot(t_gain, gain_non_normal, label="non-normal")
axes[0].axvline(peak_time, color="0.45", ls=":")
axes[0].set_xlabel("time")
axes[0].set_ylabel(r"$\|e^{At}\|_2$")
axes[0].set_title("Worst-case finite-time gain")
axes[0].legend()

axes[1].plot(t_gain, np.linalg.norm(x_normal, axis=1),
             label="normal system")
axes[1].plot(t_gain, np.linalg.norm(x_non_normal, axis=1),
             label="non-normal system")
axes[1].set_xlabel("time")
axes[1].set_ylabel(r"$\|x(t)\|_2$")
axes[1].set_title("Response from the optimizing initial state")
axes[1].legend()

fig.tight_layout()
plt.show()

The eigenvalues correctly predict that both responses eventually decay. They do not predict the magnitude of the largest transient. Singular values of the state-transition matrix quantify that finite-time amplification.

## 10. The Laplace resolvent

The resolvent

$$
R(s;A)=(sI-A)^{-1}
$$

is the Laplace transform of $e^{At}$. Its norm indicates how strongly the dynamics can amplify forcing at a complex frequency. The calculation below verifies the inverse identity and plots the resolvent norm along the imaginary axis.

In [ ]:
s_test = 1.0 + 1.5j
resolvent = np.linalg.inv(s_test * np.eye(2) - A)
print("Resolvent identity error:",
      norm((s_test * np.eye(2) - A) @ resolvent - np.eye(2)))

omega = np.linspace(-5.0, 5.0, 801)
resolvent_norm = np.array([
    svdvals(np.linalg.inv(1j * w * np.eye(2) - A))[0]
    for w in omega
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(omega, resolvent_norm)
ax.set_xlabel(r"frequency $\omega$")
ax.set_ylabel(r"$\|(j\omega I-A)^{-1}\|_2$")
ax.set_title("Resolvent norm of the damped oscillator")
plt.show()

## 11. Student investigations

1. Replace the initial condition of the oscillator and verify that the numerical errors remain small.
2. Change the damping coefficient from $0.6$ to $0.1$ and then to $1.5$. Compare the response and resolvent norm.
3. Replace the sinusoidal input with a unit step or a short pulse.
4. Increase the off-diagonal entry of the non-normal matrix from $8$ to $12$. How do the peak gain and peak time change?
5. Check numerically that $A(t_1)A(t_2)\ne A(t_2)A(t_1)$ for the LTV example.
6. Compare the LTV transition matrix with the incorrect shortcut $\exp(\int_{t_0}^{t}A(\tau)d\tau)$.

## 12. Takeaways

- The homogeneous LTI solution is $e^{A(t-t_0)}x_0$.
- The forced response is the convolution of the input with the state-transition kernel.
- Zero-input and zero-state responses add because the system is linear.
- The resolvent $(sI-A)^{-1}$ is the Laplace-domain counterpart of the matrix exponential.
- A state-transition matrix satisfies identity, inverse, composition, and matrix differential equations.
- For LTI systems, $\Phi(t,t_0)=e^{A(t-t_0)}$.
- Stable eigenvalues describe asymptotic decay but do not exclude non-normal transient growth.